# 🧩 Esercizi Guidati — Agenti AI con Tool Calling

---

Questo notebook contiene **3 esercizi progressivi**.
Ogni esercizio fornisce:
- 🎯 L'**obiettivo** da raggiungere
- 📌 I **suggerimenti** passo per passo
- ✏️ Le **celle di codice** dove scrivere la soluzione

| Esercizio | Titolo | Difficoltà |
|---|---|---|
| **1** | Il tuo primo Tool personalizzato | 🟢 Facile |
| **2** | Agente con tool di ricerca e calcolo | 🟡 Medio |
| **3** | Mini-agente conversazionale multi-turno | 🔴 Avanzato |

> **Requisiti:** Ollama in esecuzione su `localhost:11434` con modello `llama3.2` o `qwen2.5`.

---
## ⚙️ Setup — Esegui questa cella prima di iniziare

In [2]:
%pip install -q langchain langchain-ollama langchain-core


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import json
from typing import Any, List, Dict, Optional
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool

MODEL_NAME = "qwen2.5:latest"  # oppure 'qwen2.5'
BASE_URL = "http://localhost:11434"

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)

try:
    test_response = llm.invoke("Rispondi solo con: 'Pronto!'")
    print(f"✅ Connessione riuscita: {test_response.content.strip()}")
except Exception as e:
    print(f"❌ Errore di connessione: {e}")

✅ Connessione riuscita: Pronto!


In [4]:
# Funzione di utilità per visualizzare la conversazione (identica al notebook 03)
def mostra_conversazione(messaggi: List[BaseMessage]):
    print("=" * 70)
    print("📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE")
    print("=" * 70)
    for i, msg in enumerate(messaggi, 1):
        if isinstance(msg, SystemMessage):
            print(f"\n[{i}] ⚙️ SYSTEM:\n    {msg.content}")
        elif isinstance(msg, HumanMessage):
            print(f"\n[{i}] 👤 UTENTE:\n    {msg.content}")
        elif isinstance(msg, AIMessage):
            print(f"\n[{i}] 🤖 ASSISTENTE (AI):")
            if msg.content:
                print(f"    {msg.content}")
            if getattr(msg, 'tool_calls', None):
                for tc in msg.tool_calls:
                    print(f"    🛠️ [Tool Call] Funzione: '{tc['name']}' | Parametri: {json.dumps(tc['args'], ensure_ascii=False)}")
        elif isinstance(msg, ToolMessage):
            print(f"\n[{i}] 📥 RISULTATO TOOL (ID: {msg.tool_call_id}):\n    {msg.content}")
        else:
            print(f"\n[{i}] ❓ ALTRO ({type(msg).__name__}):\n    {msg.content}")
    print("\n" + "=" * 70)

---
## 🟢 Esercizio 1 — Il tuo primo Tool personalizzato

### 🎯 Obiettivo
Creare un tool `conta_parole` che:
1. Riceve un testo come stringa.
2. Conta il numero di parole nel testo.
3. Restituisce un messaggio con il conteggio.

Poi collegarlo al modello con `.bind_tools(...)` e testarlo chiedendo all'LLM di contare le parole in una frase.

### 📌 Suggerimenti
- Usa il decoratore `@tool` su una funzione Python con type annotations.
- Per contare le parole puoi usare `len(testo.split())`.
- La docstring è **fondamentale**: l'LLM la legge per capire quando usare il tool!
- Dopo aver definito il tool, usa `llm.bind_tools([conta_parole])` per collegarlo.
- Invoca il modello con una `HumanMessage` che richieda il conteggio delle parole.
- Controlla se la risposta contiene `tool_calls` e, in tal caso, esegui la funzione manualmente.

In [5]:
# 📝 ESERCIZIO 1 — Parte A: Definisci il tool `conta_parole`

# TODO: Definisci qui la funzione `conta_parole` con il decoratore @tool
# La funzione deve:
#   - Accettare un parametro 'testo' di tipo str
#   - Restituire una stringa con il conteggio delle parole
#   - Avere una docstring descrittiva

# Scrivi il tuo codice qui:

@tool
def conta_parole(testo: str) -> str:
    """Conta le parole nel testo fornito e restituisce il numero trovato."""
    numero_parole = len(testo.split())
    return f"Il testo contiene {numero_parole} parole."

In [7]:
# 📝 ESERCIZIO 1 — Parte B: Collega il tool al modello e testa

# TODO 1: Collega il tool al modello con bind_tools
# llm_con_conta = ...

# TODO 2: Crea un messaggio utente che chiede di contare le parole in:
#   "L'intelligenza artificiale sta trasformando il mondo del lavoro"
# prompt = ...
# messaggi = ...

# TODO 3: Invoca il modello e stampa la risposta
# risposta = ...
# print(...)

# Scrivi il tuo codice qui:

llm_con_conta = llm.bind_tools([conta_parole])

prompt_utente = (
    "Usa il tool conta_parole per contare le parole in questa frase: "
    "\"L'intelligenza artificiale sta trasformando il mondo del lavoro\"."
)
messaggi: List[BaseMessage] = [HumanMessage(content=prompt_utente)]

# Prima chiamata al modello
risposta = llm_con_conta.invoke(messaggi)

print("Tipo di risposta:", type(risposta))
print("Contenuto testuale:", risposta.content or "(Nessun testo, ha generato una Tool Call!)")
print("Tool calls rilevate:", risposta.tool_calls)

Tipo di risposta: <class 'langchain_core.messages.ai.AIMessage'>
Contenuto testuale: (Nessun testo, ha generato una Tool Call!)
Tool calls rilevate: [{'name': 'conta_parole', 'args': {'testo': "L'intelligenza artificiale sta trasformando il mondo del lavoro"}, 'id': '111c8305-bd42-4dd6-9199-f123849f64d8', 'type': 'tool_call'}]


In [9]:
# 📝 ESERCIZIO 1 — Parte C: Esegui il tool e ottieni la risposta finale

# TODO: Se il modello ha generato tool_calls, esegui il tool e
#       invia il ToolMessage al modello per ottenere la risposta finale.
#
# Struttura suggerita:
#   1. Aggiungi la risposta AI ai messaggi
#   2. Per ogni tool_call in risposta.tool_calls:
#       a. Estrai nome, argomenti e id
#       b. Esegui la funzione: conta_parole.invoke(argomenti)
#       c. Crea un ToolMessage con il risultato
#       d. Aggiungi il ToolMessage ai messaggi
#   3. Richiama il modello per la risposta finale
#   4. Mostra la conversazione con mostra_conversazione()

# Scrivi il tuo codice qui:

# Dizionario per mappare il nome del tool alla funzione Python
tools_map = {"conta_parole": conta_parole}

# 1. Aggiungiamo la risposta del modello (con la tool call) alla cronologia
messaggi.append(risposta)

# 2. Se il modello ha richiesto di eseguire un tool
if risposta.tool_calls:
    for tool_call in risposta.tool_calls:
        nome_tool = tool_call["name"]
        argomenti = tool_call["args"]
        call_id = tool_call.get("id", "call_1")
        
        print(f"⚙️ Esecuzione tool Python '{nome_tool}' con argomenti:")
        print(json.dumps(argomenti, indent=2, ensure_ascii=False))
        
        # Esecuzione effettiva della funzione Python
        funzione_selezionata = tools_map[nome_tool]
        risultato_tool = funzione_selezionata.invoke(argomenti)
        
        print(f"📥 Risultato ottenuto dal tool: {risultato_tool}")
        
        # 3. Inviamo l'osservazione (ToolMessage) al modello
        messaggi.append(ToolMessage(content=str(risultato_tool), tool_call_id=call_id))
    
    # 4. Chiamata al modello per formulare la risposta dopo l'esecuzione del tool
    risposta_ai_post_tool = llm_con_conta.invoke(messaggi)
    messaggi.append(risposta_ai_post_tool)
    
    print("\n💬 Risposta dell'AI all'utente:")
    print(risposta_ai_post_tool.content)
else:
    print("Il modello ha risposto direttamente:", risposta.content)


⚙️ Esecuzione tool Python 'conta_parole' con argomenti:
{
  "testo": "L'intelligenza artificiale sta trasformando il mondo del lavoro"
}
📥 Risultato ottenuto dal tool: Il testo contiene 8 parole.

💬 Risposta dell'AI all'utente:
La frase "L'intelligenza artificiale sta trasformando il mondo del lavoro" contiene 8 parole.


---
## 🟡 Esercizio 2 — Agente con Tool di Ricerca e Calcolo

### 🎯 Obiettivo
Creare un mini-agente che ha a disposizione **due tool**:
1. `cerca_prodotto(nome: str) -> str` — simula una ricerca in un catalogo e restituisce il prezzo di un prodotto.
2. `calcola_sconto(prezzo: float, percentuale_sconto: float) -> str` — calcola il prezzo finale dopo uno sconto.

L'agente deve essere in grado di rispondere a: _"Quanto costa un laptop dopo il 15% di sconto?"_ usando **entrambi i tool in sequenza**.

### 📌 Suggerimenti
- Il catalogo può essere un semplice dizionario Python all'interno del tool.
- Il tool `cerca_prodotto` dovrebbe gestire il caso in cui il prodotto non esiste.
- Usa la funzione `esegui_agente` (che devi implementare qui) basandoti sul modello del notebook 03.
- Il loop agentico deve continuare finché l'AI non genera una risposta senza `tool_calls`.

In [10]:
# 📝 ESERCIZIO 2 — Parte A: Definisci i due tool

# TODO 1: Definisci `cerca_prodotto(nome: str) -> str`
# Catalogo di esempio (puoi usarlo dentro il tool):
# CATALOGO = {
#     "laptop": 1200.0,
#     "smartphone": 699.0,
#     "tablet": 450.0,
#     "cuffie": 89.0,
#     "monitor": 320.0,
# }

# TODO 2: Definisci `calcola_sconto(prezzo: float, percentuale_sconto: float) -> str`
# Formula: prezzo_finale = prezzo * (1 - percentuale_sconto / 100)

# Scrivi il tuo codice qui:

CATALOGO = {
    "laptop": 1200.0,
    "smartphone": 699.0,
    "tablet": 450.0,
    "cuffie": 89.0,
    "monitor": 320.0,
}

@tool
def cerca_prodotto(nome: str) -> str:
    """Cerca un prodotto nel catalogo e ne restituisce il prezzo in euro.

    Args:
        nome: Nome del prodotto da cercare, ad esempio "laptop" o "tablet".

    Returns:
        Il prezzo del prodotto se presente nel catalogo, oppure un messaggio
        che indica che il prodotto non è stato trovato.
    """
    nome_normalizzato = nome.strip().lower()
    prezzo = CATALOGO.get(nome_normalizzato)

    if prezzo is None:
        return f"Prodotto '{nome}' non trovato nel catalogo."

    return f"Il prezzo di {nome_normalizzato} è {prezzo:.2f} euro."



@tool
def calcola_sconto(prezzo: float, percentuale_sconto: float) -> str:
    """Calcola il prezzo finale dopo aver applicato uno sconto percentuale.

    Args:
        prezzo: Prezzo originale del prodotto in euro.
        percentuale_sconto: Percentuale di sconto da applicare, da 0 a 100.

    Returns:
        Il prezzo finale in euro, oppure un messaggio di errore se i valori
        inseriti non sono validi.
    """
    if prezzo < 0:
        return "Errore: il prezzo non può essere negativo."

    if not 0 <= percentuale_sconto <= 100:
        return "Errore: lo sconto deve essere compreso tra 0 e 100."

    prezzo_finale = prezzo * (1 - percentuale_sconto / 100)
    return f"Prezzo finale dopo lo sconto: {prezzo_finale:.2f} euro."

In [11]:
# 📝 ESERCIZIO 2 — Parte B: Implementa la funzione `esegui_agente`

def esegui_agente(
    richiesta_utente: str,
    lista_tool: list,
    cronologia: Optional[List[BaseMessage]] = None,
    max_iterazioni: int = 6,
    verbose: bool = True,
) -> tuple[str, List[BaseMessage]]:
    """
    Esegue un loop agentico completo (Reasoning Loop / ReAct pattern).

    Il loop si chiude quando l'AI fornisce una risposta senza tool_calls,
    oppure al raggiungimento del numero massimo di iterazioni.

    Returns:
        Una tupla (risposta_finale_testo, cronologia_aggiornata)
    """
    # Creiamo il mappa nome_tool → funzione per l'esecuzione
    map_tool = {t.name: t for t in lista_tool}

    # Colleghiamo i tool al modello
    llm_con_tool = llm.bind_tools(lista_tool)

    # Inizializzazione cronologia: se non ne abbiamo una, partiamo da zero
    if cronologia is None:
        system_prompt = (
            "Usa un solo tool per volta. "
            "Dopo aver ricevuto l'output del tool, decidi il passo successivo. "
            "Non pianificare più tool contemporaneamente. Non chiedere spiegazioni ulteriore all'utente ma agisci."
        )
        messaggi: List[BaseMessage] = [SystemMessage(content=system_prompt)]
    else:
        # Copiamo la cronologia esistente per non mutare la lista originale
        messaggi = list(cronologia)

    # Aggiungiamo il nuovo messaggio dell'utente
    messaggi.append(HumanMessage(content=richiesta_utente))

    if verbose:
        print(f"🎯 NUOVA RICHIESTA: {richiesta_utente}\n" + "=" * 60)

    # ---- REASONING LOOP ----
    for step in range(1, max_iterazioni + 1):
        if verbose:
            print(f"\n📍 [Ciclo {step}] L'agente sta ragionando...")

        # Invocazione del modello con la cronologia corrente
        risposta = llm_con_tool.invoke(messaggi)
        messaggi.append(risposta)

        # Condizione di uscita: nessuna tool call → risposta finale
        if not risposta.tool_calls:
            if verbose:
                print("🏁 L'agente ha completato il task!")
            return risposta.content, messaggi

        # Esecuzione del primo tool richiesto (un tool per ciclo)
        tc = risposta.tool_calls[0]
        t_name = tc["name"]
        t_args = tc["args"]
        t_id = tc.get("id", f"call_{step}")

        if verbose:
            print(f"  ⚡ Tool '{t_name}' → argomenti: {t_args}")

        # Lookup nella mappa e invocazione
        if t_name in map_tool:
            risultato = map_tool[t_name].invoke(t_args)
        else:
            risultato = f"Errore: Tool '{t_name}' non trovato tra quelli disponibili."

        if verbose:
            print(f"  👁️ Osservazione: {risultato}")

        # Aggiungiamo il risultato come ToolMessage per il ciclo successivo
        messaggi.append(ToolMessage(content=str(risultato), tool_call_id=t_id))

    # Limite iterazioni raggiunto
    return "⚠️ Limite massimo di iterazioni raggiunto senza risposta finale.", messaggi


print("✅ Funzione 'esegui_agente' definita.")

✅ Funzione 'esegui_agente' definita.


In [12]:
# 📝 ESERCIZIO 2 — Parte C: Testa l'agente

MODEL_NAME = "qwen2.5"  # Modello più potente raccomandato

llm = ChatOllama(
    model=MODEL_NAME,
    base_url=BASE_URL,
    temperature=0,
)

# TODO: Usa la funzione esegui_agente con i due tool creati sopra per rispondere a:
# "Quanto costa un laptop dopo il 15% di sconto?"
#
# Poi prova anche con:
# "Qual è il prezzo di uno smartphone con il 20% di sconto?"

lista_tool_es2 = [cerca_prodotto, calcola_sconto]  # aggiorna con i nomi dei tuoi tool

# Scrivi il tuo codice qui:

richiesta_1 = "Quanto costa un laptop dopo il 15% di sconto?"
risposta_1, cronologia_1 = esegui_agente(
    richiesta_1,
    lista_tool_es2,
)
print("\nRisposta:", risposta_1)

richiesta_2 = "Qual è il prezzo di uno smartphone con il 20% di sconto?"
risposta_2, cronologia_2 = esegui_agente(
    richiesta_2,
    lista_tool_es2,
)
print("\nRisposta:", risposta_2)


🎯 NUOVA RICHIESTA: Quanto costa un laptop dopo il 15% di sconto?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'cerca_prodotto' → argomenti: {'nome': 'laptop'}
  👁️ Osservazione: Il prezzo di laptop è 1200.00 euro.

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Tool 'calcola_sconto' → argomenti: {'prezzo': 1200, 'percentuale_sconto': 15}
  👁️ Osservazione: Prezzo finale dopo lo sconto: 1020.00 euro.

📍 [Ciclo 3] L'agente sta ragionando...
🏁 L'agente ha completato il task!

Risposta: Il prezzo finale del laptop dopo il 15% di sconto è di 1020.00 euro.
🎯 NUOVA RICHIESTA: Qual è il prezzo di uno smartphone con il 20% di sconto?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'cerca_prodotto' → argomenti: {'nome': 'smartphone'}
  👁️ Osservazione: Il prezzo di smartphone è 699.00 euro.

📍 [Ciclo 2] L'agente sta ragionando...
  ⚡ Tool 'calcola_sconto' → argomenti: {'prezzo': 699, 'percentuale_sconto': 20}
  👁️ Osservazione: Prezzo finale dopo lo sconto: 559.20 euro.

📍 [Ciclo 3] L'agente st

---
## 🔴 Esercizio 3 — Mini-Agente Conversazionale Multi-Turno

### 🎯 Obiettivo
Costruire un agente che:
1. Ha a disposizione **tre tool**: `scrivi_nota`, `leggi_nota`, `lista_note`.
2. Mantiene la **memoria della conversazione** tra i turni (passando la cronologia tra le chiamate).
3. Completa una sequenza di **3 operazioni in conversazioni separate**:
   - **Turno 1**: Scrivere una nota su "Python" con un breve contenuto.
   - **Turno 2**: Leggere la nota appena creata (l'agente deve ricordare il contesto del turno 1).
   - **Turno 3**: Chiedere all'agente la lista di tutte le note esistenti.

### 📌 Suggerimenti
- Le note possono essere salvate come file `.txt` in una cartella `note/`.
- `scrivi_nota(titolo: str, contenuto: str)` → crea `note/{titolo}.txt`.
- `leggi_nota(titolo: str)` → legge `note/{titolo}.txt`.
- `lista_note()` → elenca tutti i file in `note/` (questo tool non ha argomenti obbligatori).
- La **chiave** dell'esercizio è passare `cronologia` al secondo e terzo turno per mantenere la memoria!
- Usa `mostra_conversazione()` alla fine per visualizzare l'intera storia dei 3 turni.

In [13]:
# 📝 ESERCIZIO 3 — Parte A: Definisci i tre tool per la gestione note

# Crea prima la cartella 'note' se non esiste
os.makedirs("note", exist_ok=True)

# TODO 1: Definisci `scrivi_nota(titolo: str, contenuto: str) -> str`
# Salva il file in: f"note/{titolo}.txt"

# TODO 2: Definisci `leggi_nota(titolo: str) -> str`
# Legge il file da: f"note/{titolo}.txt"
# Gestisci il caso in cui il file non esiste!

# TODO 3: Definisci `lista_note() -> str`
# Elenca tutti i file .txt nella cartella 'note/'
# Suggerimento: usa os.listdir("note") e filtra per '.txt'
# Nota: questa funzione non ha parametri (usa una stringa vuota come placeholder se necessario)

# Scrivi il tuo codice qui:

@tool
def scrivi_nota(titolo: str, contenuto: str) -> str:
    """Salva il contenuto in un file di testo nella cartella 'note'.

    Args:
        titolo: Titolo della nota, usato per creare il nome del file.
        contenuto: Testo da salvare.

    Returns:
        Un messaggio che conferma il salvataggio della nota.
    """
    percorso = f"note/{titolo}.txt"
    with open(percorso, "w", encoding="utf-8") as file:
        file.write(contenuto)
    return f"Nota '{titolo}' salvata."


@tool
def leggi_nota(titolo: str) -> str:
    """Legge il contenuto di una nota dalla cartella 'note'.

    Args:
        titolo: Titolo della nota da leggere, senza l'estensione '.txt'.

    Returns:
        Il contenuto della nota oppure un messaggio se il file non esiste.
    """
    percorso = f"note/{titolo}.txt"
    try:
        with open(percorso, "r", encoding="utf-8") as file:
            return file.read()
    except FileNotFoundError:
        return f"Nota '{titolo}' non trovata."


@tool
def lista_note() -> str:
    """Restituisce l'elenco dei file .txt presenti nella cartella 'note'.

    Non richiede argomenti.

    Returns:
        I nomi delle note disponibili oppure un messaggio se non ce ne sono.
    """
    nomi_note = [
        nome for nome in os.listdir("note")
        if nome.endswith(".txt")
    ]

    if not nomi_note:
        return "Non ci sono note salvate."

    return "Note disponibili: " + ", ".join(sorted(nomi_note))

In [14]:
# 📝 ESERCIZIO 3 — Parte B: Turno 1 — Scrivi una nota

lista_tool_es3 = [scrivi_nota, leggi_nota, lista_note]

richiesta_turno_1 = (
    "Scrivi una nota dal titolo 'Python' con questo contenuto: "
    "'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'"
)

risposta_t1, cronologia_completa = esegui_agente(
    richiesta_turno_1,
    lista_tool_es3,
)
print("\n💬 Risposta Turno 1:")
print(risposta_t1)


🎯 NUOVA RICHIESTA: Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'scrivi_nota' → argomenti: {'titolo': 'Python', 'contenuto': 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'}
  👁️ Osservazione: Nota 'Python' salvata.

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 1:
La tua nota dal titolo "Python" è stata salvata con il seguente contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.' Se vuoi, posso mostrarti l'elenco delle tue note.


In [15]:
# 📝 ESERCIZIO 3 — Parte C: Turno 2 — Leggi la nota (con memoria del Turno 1)

# TODO: Chiedi all'agente di leggere la nota 'Python'
# IMPORTANTE: passa cronologia_completa come parametro per mantenere la memoria!
#
# Richiesta: "Puoi leggere la nota che hai appena scritto su Python?"

# Scrivi il tuo codice qui:

risposta_t2, cronologia_completa = esegui_agente(
    "Puoi leggere la nota che hai appena scritto su Python?",
    lista_tool_es3,
    cronologia=cronologia_completa,  # ← qui passiamo la memoria!
)
print("\n💬 Risposta Turno 2:")
print(risposta_t2)

🎯 NUOVA RICHIESTA: Puoi leggere la nota che hai appena scritto su Python?

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'leggi_nota' → argomenti: {'titolo': 'Python'}
  👁️ Osservazione: Python è un linguaggio di programmazione versatile, molto usato in AI e data science.

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 2:
Ecco il contenuto della tua nota "Python": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."


In [16]:
# 📝 ESERCIZIO 3 — Parte D: Turno 3 — Lista di tutte le note

# TODO: Chiedi all'agente di elencare tutte le note disponibili
# (continua a passare la cronologia aggiornata)
#
# Richiesta: "Quante note hai salvato finora? Elencale."
#
# Alla fine, usa mostra_conversazione() per visualizzare
# l'intera storia dei 3 turni in un'unica cronologia.

# Scrivi il tuo codice qui:

risposta_t3, cronologia_completa = esegui_agente(
    "Quante note hai salvato finora? Elencale.",
    lista_tool_es3,
    cronologia=cronologia_completa,  # ← continuiamo ad accumulare la memoria
)
print("\n💬 Risposta Turno 3:")
print(risposta_t3)


🎯 NUOVA RICHIESTA: Quante note hai salvato finora? Elencale.

📍 [Ciclo 1] L'agente sta ragionando...
  ⚡ Tool 'lista_note' → argomenti: {}
  👁️ Osservazione: Note disponibili: Python.txt

📍 [Ciclo 2] L'agente sta ragionando...
🏁 L'agente ha completato il task!

💬 Risposta Turno 3:
Finora hai salvato una nota. Ecco l'elenco delle tue note: "Python.txt".


In [17]:
# Visualizzazione dell'intera cronologia dei 3 turni
print(f"Totale messaggi in cronologia: {len(cronologia_completa)}\n")
mostra_conversazione(messaggi=cronologia_completa)

Totale messaggi in cronologia: 13

📜 CRONOLOGIA COMPLETA DELLA CONVERSAZIONE

[1] ⚙️ SYSTEM:
    Usa un solo tool per volta. Dopo aver ricevuto l'output del tool, decidi il passo successivo. Non pianificare più tool contemporaneamente. Non chiedere spiegazioni ulteriore all'utente ma agisci.

[2] 👤 UTENTE:
    Scrivi una nota dal titolo 'Python' con questo contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.'

[3] 🤖 ASSISTENTE (AI):
    🛠️ [Tool Call] Funzione: 'scrivi_nota' | Parametri: {"titolo": "Python", "contenuto": "Python è un linguaggio di programmazione versatile, molto usato in AI e data science."}

[4] 📥 RISULTATO TOOL (ID: a3d8c312-efc6-4672-adf6-a67a168cab44):
    Nota 'Python' salvata.

[5] 🤖 ASSISTENTE (AI):
    La tua nota dal titolo "Python" è stata salvata con il seguente contenuto: 'Python è un linguaggio di programmazione versatile, molto usato in AI e data science.' Se vuoi, posso mostrarti l'elenco delle tue note.

[6] 

In [ ]:
# Verifica finale: il file esiste davvero sul disco?
percorso_nota = "note/Python.txt"
if os.path.exists(percorso_nota):
    print(f"✅ Il file '{percorso_nota}' esiste!")
    with open(percorso_nota, "r", encoding="utf-8") as f:
        print("--- Contenuto ---")
        print(f.read())
        print("-----------------")
else:
    print(f"⚠️ Il file '{percorso_nota}' non è stato trovato.")

✅ Il file 'note/Python.txt' esiste!
--- Contenuto ---
Python è un linguaggio di programmazione versatile, molto usato in AI e data science.
-----------------


: 

---
## 🏁 Ricapitolo degli Esercizi

| Esercizio | Cosa hai imparato |
|---|---|
| **1 — Tool base** | Creare un tool con `@tool`, type annotations, docstring; binding e invocazione manuale del tool |
| **2 — Multi-tool sequenziale** | Reasoning loop multi-step, map dei tool, tool che si concatenano |
| **3 — Multi-turno con memoria** | Persistenza della `cronologia` tra turni, tool senza argomenti, agente con stato |

> 💡 **Prossimi passi**: consulta il notebook `04_soluzioni_agenti_ai.ipynb` per confrontare la tua soluzione con quella di riferimento.